In [ ]:
# Definisci il callback per interrompere l'allenamento se si raggiunge una certa accuracy
ACCURACY_THRESHOLD = 0.94

class myCallback(k.callbacks.Callback):
    def on_epoch_end(self, epoch, logs={}):
        if(logs.get('val_accuracy') > ACCURACY_THRESHOLD):
            print("\n\nStopping training as we have reached %2.2f%% accuracy!" % (ACCURACY_THRESHOLD*100))
            self.model.stop_training = True

# Funzione di allenamento con calcolo accurato su test e la loss per training, validation, and test set
def trainModel(model, epochs, optimizer, X_train, y_train, X_dev, y_dev, X_test, y_test):
    batch_size = 128
    callback = myCallback()

    # Liste per memorizzare le metriche per ogni epoca
    test_accuracies = []
    test_losses = []
    train_losses = []
    val_losses = []

    # Funzione di callback per calcolare la loss e l'accuratezza sui dati di test alla fine di ogni epoca
    class TestMetricsCallback(k.callbacks.Callback):
        def on_epoch_end(self, epoch, logs={}):
            # Calcola la loss e l'accuratezza sui dati di test
            test_loss, test_acc = self.model.evaluate(X_test, y_test, verbose=0)
            test_losses.append(test_loss)
            test_accuracies.append(test_acc)
            print(f"Epoch {epoch+1}: Test Loss = {test_loss:.4f}, Test Accuracy = {test_acc*100:.2f}%")
        
    # Aggiungi il callback per monitorare la loss e l'accuratezza sui dati di test
    test_metrics_callback = TestMetricsCallback()

    # Compilazione del modello
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Addestramento del modello
    history = model.fit(X_train, y_train, validation_data=(X_dev, y_dev), epochs=epochs,
                        batch_size=batch_size, callbacks=[callback, test_metrics_callback])

    # Salva la loss di training e validation per ogni epoca
    train_losses = history.history['loss']
    val_losses = history.history['val_loss']

    # Ritorna l'oggetto storico e le metriche sui dati di test
    return history, test_accuracies, test_losses, train_losses, val_losses

# Funzione per tracciare le metriche (Accuracy e Loss)
def plotHistory(history, test_accuracies, test_losses, train_losses, val_losses):
    # Stampa l'accuratezza massima sulla validazione
    print("Max. Validation Accuracy", max(history.history["val_accuracy"]))
    


    # Crea un DataFrame per i dati storici
    history_df = pd.DataFrame(history.history)
    
    # Aggiungi le metriche per il test set nel DataFrame
    history_df['test_accuracy'] = test_accuracies
    history_df['test_loss'] = test_losses
    history_df['train_loss'] = train_losses
    history_df['val_loss'] = val_losses

    # Traccia il grafico delle metriche
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Assegna l'asse Y di sinistra (per Accuracy)
    ax1.plot(history_df['accuracy'], label='Train Accuracy', color='tab:blue')
    ax1.plot(history_df['val_accuracy'], label='Validation Accuracy', color='tab:orange')
    ax1.plot(history_df['test_accuracy'], label='Test Accuracy', color='tab:green')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Accuracy', color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')

    # Crea un secondo asse Y condiviso per la loss
    ax2 = ax1.twinx()
    ax2.plot(history_df['train_loss'], label='Train Loss', color='tab:red')
    ax2.plot(history_df['val_loss'], label='Validation Loss', color='tab:purple')
    ax2.plot(history_df['test_loss'], label='Test Loss', color='tab:brown')
    ax2.set_ylabel('Loss', color='tab:red')
    ax2.tick_params(axis='y', labelcolor='tab:red')

    # Aggiungi la leggenda
    fig.tight_layout()
    ax1.legend(loc='upper left')
    ax2.legend(loc='upper right')

    plt.title('Training, Validation and Test Accuracy & Loss')
    plt.show()

In [ ]:
model_1 = k.models.Sequential([
    k.layers.Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    k.layers.Dense(128, activation='relu'),
    k.layers.Dense(64, activation='relu'),
    k.layers.Dense(10, activation='softmax'),
])
print(model_1.summary())
model_1_history, test_accuracies, test_losses, train_losses, val_losses = trainModel(
    model=model_1, epochs=70, optimizer='adam', 
    X_train=X_train, y_train=y_train, 
    X_dev=X_dev, y_dev=y_dev, 
    X_test=X_test, y_test=y_test
)


In [ ]:
plotHistory(model_1_history, test_accuracies, test_losses, train_losses, val_losses)
test_loss, test_acc  = model_1.evaluate(X_test, y_test, batch_size=128)
print("The test Loss is :",test_loss)
print("\nThe Best test Accuracy is :",test_acc*100)